<a href="https://colab.research.google.com/github/OdysseusPolymetis/enexdi_prep_2026/blob/main/4_transformers_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction simple aux Transformers pour les SHS

Ce notebook propose une introduction très simple aux modèles de type **Transformer** avec la bibliothèque `transformers` de Hugging Face.

L’objectif n’est pas d’entraîner un modèle, mais de voir ce qu’on peut faire avec des modèles déjà entraînés.

On utilise principalement des implémentations par défaut avec `pipeline()`.


## 1. Installation

In [ ]:
!pip -q install transformers sentencepiece accelerate pandas scikit-learn matplotlib

## 2. Imports et configuration

On indique simplement à `transformers` d’utiliser le GPU si disponible.


In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForMaskedLM

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

device = 0 if torch.cuda.is_available() else -1

## 3. Charger un modèle multilingue : XLM-RoBERTa

Nous commençons par charger **XLM-RoBERTa**, un modèle multilingue.

Il peut être utilisé directement pour une tâche de type **fill-mask** : on donne une phrase avec un mot masqué, et le modèle propose des complétions probables.

Le masque utilisé par XLM-RoBERTa est :

```text
<mask>
```


In [ ]:
modele_xlmr = "FacebookAI/xlm-roberta-base"

tokenizer_xlmr = AutoTokenizer.from_pretrained(modele_xlmr)
model_xlmr = AutoModelForMaskedLM.from_pretrained(modele_xlmr)

fill_mask = pipeline(
    "fill-mask",
    model=model_xlmr,
    tokenizer=tokenizer_xlmr,
    device=device
)

print("Modèle chargé :", modele_xlmr)

## 4. Tâche 1 — Fill-mask

Le modèle doit compléter une phrase dans laquelle un mot est remplacé par `<mask>`.

Essayez de modifier les exemples.


In [ ]:
phrase = "Paris est la capitale de la <mask>."

fill_mask(phrase, top_k=10)

## 5. Exercice — Comparer plusieurs phrases masquées

**Consigne :** modifiez les phrases ci-dessous, puis observez les réponses.

Questions possibles :

- les réponses sont-elles pertinentes ?
- les résultats changent-ils selon la formulation ?
- les résultats changent-ils selon la langue ?
- quels présupposés le modèle semble-t-il avoir appris ?


In [ ]:
phrases = [
    "La Révolution française commence en <mask>.",
    "Le personnage principal du roman est un <mask>.",
    "The capital of France is <mask>.",
    "La ville de Lyon se trouve en <mask>."
]

for phrase in phrases:
    print("\nPhrase :", phrase)
    resultats = fill_mask(phrase, top_k=5)
    for r in resultats:
        print(r["token_str"], "→", round(r["score"], 4))

## 6. Tâche 2 — Classification zero-shot

La classification *zero-shot* permet de classer un texte dans des catégories que l’on définit soi-même, sans entraîner le modèle sur nos propres données.

Attention : cette tâche nécessite un modèle spécialisé dans l’inférence textuelle (*NLI*).  
On charge donc un second modèle, multilingue, adapté à cette tâche.


In [ ]:
modele_zero_shot = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"

zero_shot = pipeline(
    "zero-shot-classification",
    model=modele_zero_shot,
    device=device
)

print("Modèle chargé :", modele_zero_shot)

## 7. Classer un texte avec des catégories choisies

On fournit :

- un texte ;
- une liste d’étiquettes possibles.

Le modèle renvoie un score pour chaque étiquette.


In [ ]:
texte = """
Le gouvernement annonce une nouvelle réforme visant à modifier l'organisation des universités
et le financement de la recherche publique.
"""

etiquettes = [
    "politique",
    "économie",
    "religion",
    "littérature",
    "science",
    "éducation"
]

zero_shot(texte, etiquettes)

## 8. Classer un texte avec plusieurs étiquettes possibles

Avec `multi_label=True`, plusieurs catégories peuvent être considérées comme pertinentes en même temps.


In [ ]:
zero_shot(
    texte,
    etiquettes,
    multi_label=True
)

## 9. Exercice — Classer plusieurs textes courts

Voici un petit corpus fictif.

**Consigne :** modifiez les textes ou les étiquettes pour les adapter à votre domaine : histoire, littérature, sociologie, presse, archives, etc.


In [ ]:
corpus = [
    "Le roi reçoit les ambassadeurs dans la grande salle du palais.",
    "Le narrateur décrit longuement les sentiments amoureux du personnage.",
    "Les ouvriers se mettent en grève pour demander une augmentation des salaires.",
    "Le savant observe les astres et rédige un traité sur le mouvement des planètes.",
    "Le sermon insiste sur la faute, le pardon et le salut des âmes."
]

etiquettes = [
    "politique",
    "amour",
    "travail",
    "science",
    "religion"
]

resultats = []

for texte in corpus:
    prediction = zero_shot(texte, etiquettes)

    resultats.append({
        "texte": texte,
        "meilleure_etiquette": prediction["labels"][0],
        "score": prediction["scores"][0]
    })

df = pd.DataFrame(resultats)
df

## 10. Exercice — Changer les étiquettes

La classification dépend fortement des étiquettes proposées.

Essayez de remplacer les étiquettes ci-dessous par des catégories plus fines.


In [ ]:
nouvelles_etiquettes = [
    "pouvoir royal",
    "sentiment amoureux",
    "conflit social",
    "savoir scientifique",
    "discours religieux"
]

resultats = []

for texte in corpus:
    prediction = zero_shot(texte, nouvelles_etiquettes)

    resultats.append({
        "texte": texte,
        "meilleure_etiquette": prediction["labels"][0],
        "score": prediction["scores"][0]
    })

pd.DataFrame(resultats)

## 11. Tâche 3 — Représenter des phrases par des vecteurs

Un modèle Transformer peut aussi produire des vecteurs.

Ici, on utilise le même modèle XLM-RoBERTa chargé plus haut.

L’idée est simple :

1. on transforme une phrase en tokens ;
2. le modèle produit un vecteur pour chaque token ;
3. on fait une moyenne pour obtenir un vecteur de phrase.

Ce n’est pas la meilleure méthode possible pour tous les usages, mais elle est suffisante pour comprendre le principe.


In [ ]:
model_xlmr.to("cuda" if torch.cuda.is_available() else "cpu")

def vectoriser_phrase(phrase):
    inputs = tokenizer_xlmr(
        phrase,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_xlmr.roberta(**inputs)

    embeddings_tokens = outputs.last_hidden_state[0]
    attention_mask = inputs["attention_mask"][0]

    embeddings_tokens = embeddings_tokens[attention_mask == 1]

    vecteur_phrase = embeddings_tokens.mean(dim=0)

    return vecteur_phrase.cpu().numpy()

## 12. Comparer deux phrases

Plus le score est proche de 1, plus les deux phrases sont proches dans l’espace vectoriel du modèle.


In [ ]:
phrase_1 = "Le roi gouverne le royaume."
phrase_2 = "Le souverain dirige le pays."

v1 = vectoriser_phrase(phrase_1)
v2 = vectoriser_phrase(phrase_2)

score = cosine_similarity([v1], [v2])[0][0]

print("Similarité :", score)

## 13. Exercice — Comparer plusieurs phrases

Modifiez les phrases et observez les scores.


In [ ]:
phrases_a_comparer = [
    "Le roi gouverne le royaume.",
    "Le souverain dirige le pays.",
    "La jeune femme écrit une lettre d'amour.",
    "Les ouvriers réclament de meilleurs salaires.",
    "Le chercheur analyse des données linguistiques."
]

vecteurs = [vectoriser_phrase(p) for p in phrases_a_comparer]

matrice_similarite = cosine_similarity(vecteurs)

df_sim = pd.DataFrame(
    matrice_similarite,
    index=phrases_a_comparer,
    columns=phrases_a_comparer
)

df_sim

## 14. Visualiser les phrases en deux dimensions

On réduit les vecteurs à deux dimensions avec une PCA pour produire une visualisation simple.


In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(vecteurs)

plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1])

for phrase, x, y in zip(phrases_a_comparer, coords[:, 0], coords[:, 1]):
    plt.text(x, y, phrase[:35] + "...", fontsize=9)

plt.title("Projection simple de phrases")
plt.xlabel("Axe 1")
plt.ylabel("Axe 2")
plt.show()

## 15. Mini-activité SHS — Observer les effets de formulation

Les modèles ne lisent pas les textes comme des humains.  
Ils répondent à partir de régularités apprises dans de très grands corpus.

**Consigne :** testez plusieurs formulations proches.


In [ ]:
formulations = [
    "Ce texte parle de la Révolution française.",
    "Ce document évoque des événements politiques en France.",
    "Cette source historique décrit une période de crise sociale.",
    "Le passage concerne les sentiments d'un personnage romanesque."
]

etiquettes = [
    "histoire politique",
    "histoire sociale",
    "analyse littéraire",
    "histoire religieuse"
]

for texte in formulations:
    prediction = zero_shot(texte, etiquettes)
    print("\nTexte :", texte)
    print("Classe proposée :", prediction["labels"][0])
    print("Score :", round(prediction["scores"][0], 4))

## 16. Classification de tokens


In [ ]:
ner = pipeline(
    "token-classification",
    model="Davlan/xlm-roberta-base-ner-hrl",
    aggregation_strategy="simple",
    device=device
)

texte = """
Victor Hugo naît à Besançon en 1802. Il séjourne à Paris et publie Notre-Dame de Paris en 1831.
"""

ner(texte)

## 17. Classification de sentiments

In [ ]:
sentiment = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    device=device
)

phrases = [
    "Ce roman est magnifique.",
    "Cette situation est catastrophique.",
    "Voilà une décision admirablement absurde."
]

sentiment(phrases)